In [ ]:
!pip install opencv-python scikit-image tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import cv2
from tqdm import tqdm
from skimage.metrics import structural_similarity as ssim

In [ ]:
VIDEO_FOLDER = "/content/drive/MyDrive/BanglaVision/BD Dataset Videos"

OUTPUT_FOLDER = "/content/drive/MyDrive/bd_images"
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

print("Video Folder:", VIDEO_FOLDER)

In [ ]:
files = os.listdir(VIDEO_FOLDER)

video_files = [f for f in files if f.endswith((".mp4", ".avi", ".mov"))]

print("Videos found:", len(video_files))
print(video_files)

In [ ]:
def extract_frames_clean(video_path, output_folder, time_interval=1, ssim_threshold=0.90):

    cap = cv2.VideoCapture(video_path)

    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_interval = int(fps * time_interval)

    video_name = os.path.splitext(os.path.basename(video_path))[0]
    video_output = os.path.join(output_folder, video_name)

    os.makedirs(video_output, exist_ok=True)

    count = 0
    saved = 0
    prev_gray = None

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        if count % frame_interval == 0:

            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

            if prev_gray is not None:
                score = ssim(gray, prev_gray)

                if score > ssim_threshold:
                    count += 1
                    continue

            filename = f"{video_name}_{saved}.jpg"
            save_path = os.path.join(video_output, filename)

            cv2.imwrite(save_path, frame)

            prev_gray = gray
            saved += 1

        count += 1

    cap.release()

    return saved

In [ ]:
total_images = 0

for video in tqdm(video_files):

    video_path = os.path.join(VIDEO_FOLDER, video)

    print(f"\nProcessing: {video}")

    saved = extract_frames_clean(
        video_path,
        OUTPUT_FOLDER,
        time_interval=1,
        ssim_threshold=0.90
    )

    print(f"Saved frames: {saved}")
    total_images += saved

print("\nTOTAL IMAGES EXTRACTED:", total_images)